## Import Libraries

In [43]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import category_encoders as ce

## Load the Dataset

In [60]:
df = pd.read_csv("../data/clean_data.csv")
df.head()

,match_result,home_club,away_club,fixture_date,competition,competition_code,cup_game,home_manager_id,away_manager_id,home_club_recent_fixture_date_1,...,away_club_recent_competition_code_1,away_club_recent_competition_code_2,away_club_recent_competition_code_3,away_club_recent_competition_code_4,away_club_recent_competition_code_5,away_club_recent_competition_code_6,away_club_recent_competition_code_7,away_club_recent_competition_code_8,away_club_recent_competition_code_9,away_club_recent_competition_code_10
0,away,Newell's Old Boys,River Plate,2019-12-01 00:45:00,Superliga,636,0,468196.0,468200.0,2019-11-26 00:10:00,...,1122.0,642.0,636.0,636.0,636.0,1122.0,636.0,642.0,636.0,1122.0
1,home,Real Estelí,Deportivo Las Sabanas,2019-12-01 01:00:00,Primera Division,752,0,516788.0,22169161.0,2019-11-27 21:00:00,...,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0,752.0
2,draw,UPNFM,Marathón,2019-12-01 01:00:00,Liga Nacional,734,0,2510608.0,456313.0,2019-11-28 01:15:00,...,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0,734.0
3,away,León,Morelia,2019-12-01 01:00:00,Liga MX,743,0,1552508.0,465797.0,2019-11-28 01:00:00,...,743.0,743.0,743.0,743.0,743.0,743.0,743.0,743.0,746.0,743.0
4,home,Cobán Imperial,Iztapa,2019-12-01 01:00:00,Liga Nacional,705,0,429958.0,426870.0,2019-11-27 18:00:00,...,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0,705.0


## Time-Based Feature Extraction
The `fixture_date` column is used to extract time-based features such as year, month, and day of the week. 

Before extracting these features, the dates are standardized to UTC. Crucially, the dataset is then sorted chronologically. This completely prevents future data leakage when training our models, ensuring that past data strictly predicts future data.

In [45]:
# Convert to datetime and standardize to UTC
df['fixture_date'] = pd.to_datetime(df['fixture_date'], utc=True)

# Sort chronologically to prevent data leakage
df = df.sort_values('fixture_date').reset_index(drop=True)

# Extract time-based features
df['fixture_year'] = df['fixture_date'].dt.year
df['fixture_month'] = df['fixture_date'].dt.month
df['fixture_day_of_week'] = df['fixture_date'].dt.dayofweek

# Create a season indicator (differentiating summer break months)
df['is_summer_break'] = df['fixture_month'].isin([6, 7]).astype(int)

## Schedule Congestion and Fatigue Indicators
Machine learning models cannot process raw timestamp strings. Furthermore, the absolute date of a previous match is less important than the relative time elapsed. 

Here, we convert the 10 historical fixture dates into a "days since match" numerical feature for both teams. This mathematically represents schedule congestion and potential team fatigue. Afterward, the original raw string columns are dropped to keep the dataset model-ready.

In [46]:
def calculate_rest_days(df, team_type):
    for i in range(1, 11):
        hist_date_col = f"{team_type}_club_recent_fixture_date_{i}"
        rest_days_col = f"{team_type}_days_since_match_{i}"
        
        df[hist_date_col] = pd.to_datetime(df[hist_date_col], errors='coerce', utc=True)
        
        df[rest_days_col] = (df['fixture_date'] - df[hist_date_col]).dt.days
        
        # If a team hasn't played 10 previous matches, fill the missing rest days with 90 days
        df[rest_days_col] = df[rest_days_col].fillna(90)
        
        # Drop the original unreadable string column
        df.drop(columns=[hist_date_col], inplace=True)
        
    return df

# Apply the fatigue calculation to both home and away teams
df = calculate_rest_days(df, "home")
df = calculate_rest_days(df, "away")

df.drop(columns=['fixture_date'], inplace=True)

## Match Context Differentiation
Cup matches often have different patterns than regular competition matches. We need to ensure the `cup_game` flag is properly transformed into a strict numerical format.

In [47]:
# Convert string/boolean True/False mapping to 1/0
df['cup_game'] = df['cup_game'].astype(str).map({'True': 1, 'False': 0, '1': 1, '0': 0})

# Fill any remaining NaNs with 0 and convert to integer
df['cup_game'] = df['cup_game'].fillna(0).astype(int)

## Historical Form and Performance Indicators
To effectively describe recent club performance and consistency, we calculate aggregate statistics from the last 10 matches. We will engineer the mean goals scored, mean goals conceded, and mean team ratings over the recent rolling window for both the home and away clubs.

In [48]:
def calculate_recent_stats(df, team_type):
    goals_scored_cols = [f"{team_type}_club_recent_goals_scored_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_goals_scored'] = df[goals_scored_cols].mean(axis=1)

    goals_conceded_cols = [f"{team_type}_club_recent_goals_conceded_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_goals_conceded'] = df[goals_conceded_cols].mean(axis=1)

    rating_cols = [f"{team_type}_club_recent_team_rating_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_team_rating'] = df[rating_cols].mean(axis=1)

    # Calculate Opponent Rating
    opp_rating_cols = [f"{team_type}_club_recent_opponent_rating_{i}" for i in range(1, 11)]
    df[f'{team_type}_avg_opp_rating'] = df[opp_rating_cols].mean(axis=1)

    # True Strength Metric (Team Rating divided by Opponent Rating)
    df[f'{team_type}_true_strength'] = df[f'{team_type}_avg_team_rating'] / (df[f'{team_type}_avg_opp_rating'] + 0.01)

    return df

# Apply aggregations to both home and away teams
df = calculate_recent_stats(df, "home")
df = calculate_recent_stats(df, "away")

## Categorical Data Transformation
Categorical attributes such as club names and competition names must be transformed into a numerical format for the machine learning algorithms. We apply Label Encoding to these features, as well as to our 3-class target variable (`match_result`). Manager IDs are also strictly cast to numerical floats.

In [49]:
le = LabelEncoder()

df['match_result'] = le.fit_transform(df['match_result'].astype(str))

df.drop(columns=['home_manager_id', 'away_manager_id'], inplace=True, errors='ignore')

### Average Goals in Recent Matches

A team's recent attacking and defensive performance provides valuable information about its current form.

To summarize performance over the last ten matches, the average number of goals scored and conceded is calculated for both the home and away teams. These features provide a stable estimate of offensive and defensive strength while reducing the impact of individual matches.

In [50]:
# Average goals scored and conceded over the last 10 matches

home_scored = [f"home_club_recent_goals_scored_{i}" for i in range(1, 11)]
home_conceded = [f"home_club_recent_goals_conceded_{i}" for i in range(1, 11)]

away_scored = [f"away_club_recent_goals_scored_{i}" for i in range(1, 11)]
away_conceded = [f"away_club_recent_goals_conceded_{i}" for i in range(1, 11)]

df["home_avg_goals_scored"] = df[home_scored].mean(axis=1)
df["home_avg_goals_conceded"] = df[home_conceded].mean(axis=1)

df["away_avg_goals_scored"] = df[away_scored].mean(axis=1)
df["away_avg_goals_conceded"] = df[away_conceded].mean(axis=1)

### Team and Opponent Strength

Average team ratings over recent matches represent each team's overall quality.

Opponent ratings are also averaged to measure the difficulty of the teams recently faced. Combining these values provides a more reliable estimate of true team strength.

In [51]:
# Average team and opponent ratings

home_rating = [f"home_club_recent_team_rating_{i}" for i in range(1, 11)]
away_rating = [f"away_club_recent_team_rating_{i}" for i in range(1, 11)]

home_opp = [f"home_club_recent_opponent_rating_{i}" for i in range(1, 11)]
away_opp = [f"away_club_recent_opponent_rating_{i}" for i in range(1, 11)]

df["home_avg_team_rating"] = df[home_rating].mean(axis=1)
df["away_avg_team_rating"] = df[away_rating].mean(axis=1)

df["home_avg_opp_rating"] = df[home_opp].mean(axis=1)
df["away_avg_opp_rating"] = df[away_opp].mean(axis=1)

df["home_true_strength"] = (
    df["home_avg_team_rating"] - df["home_avg_opp_rating"]
)

df["away_true_strength"] = (
    df["away_avg_team_rating"] - df["away_avg_opp_rating"]
)

### Goal Difference

Goal difference is a simple but effective indicator of team performance.

It measures the balance between attacking and defensive ability by subtracting the average goals conceded from the average goals scored. A larger value generally indicates stronger recent performance.

In [52]:
# Goal difference

df["home_goal_difference"] = (
    df["home_avg_goals_scored"] -
    df["home_avg_goals_conceded"]
)

df["away_goal_difference"] = (
    df["away_avg_goals_scored"] -
    df["away_avg_goals_conceded"]
)

### Average Rest Days

Recovery time between matches can affect player fitness and overall team performance.

The average number of rest days over the previous ten matches is calculated to estimate player fatigue and schedule intensity.

In [53]:
# Average rest days

home_rest = [f"home_days_since_match_{i}" for i in range(1, 11)]
away_rest = [f"away_days_since_match_{i}" for i in range(1, 11)]

df["home_avg_rest_days"] = df[home_rest].mean(axis=1)
df["away_avg_rest_days"] = df[away_rest].mean(axis=1)

### Home Match Ratio

Teams often perform differently when playing at home.

The proportion of recent matches played at home is calculated to capture familiarity with home conditions and possible home-field advantage.

In [54]:
# Home match ratio

home_home = [f"home_club_recent_played_at_home_{i}" for i in range(1, 11)]
away_home = [f"away_club_recent_played_at_home_{i}" for i in range(1, 11)]

df["home_recent_home_ratio"] = df[home_home].mean(axis=1)
df["away_recent_home_ratio"] = df[away_home].mean(axis=1)

### Cup Match Ratio

Cup competitions often differ from league matches in terms of playing style and squad rotation.

The ratio of recent cup matches is calculated to represent each team's recent experience in knockout competitions.

In [55]:
# Cup match ratio

home_cup = [f"home_club_recent_cup_game_{i}" for i in range(1, 11)]
away_cup = [f"away_club_recent_cup_game_{i}" for i in range(1, 11)]

df["home_recent_cup_ratio"] = df[home_cup].mean(axis=1)
df["away_recent_cup_ratio"] = df[away_cup].mean(axis=1)

### Comparative Team Form

Rather than using only absolute statistics, relative differences between the two teams are also calculated.

These comparative features directly capture the strength gap between the home and away clubs, making them more informative for match outcome prediction.

In [56]:
# Comparative features

df["rating_difference"] = (
    df["home_avg_team_rating"] -
    df["away_avg_team_rating"]
)

df["goal_difference_difference"] = (
    df["home_goal_difference"] -
    df["away_goal_difference"]
)

df["rest_days_difference"] = (
    df["home_avg_rest_days"] -
    df["away_avg_rest_days"]
)

df["home_ratio_difference"] = (
    df["home_recent_home_ratio"] -
    df["away_recent_home_ratio"]
)

df["cup_ratio_difference"] = (
    df["home_recent_cup_ratio"] -
    df["away_recent_cup_ratio"]
)

df["true_strength_difference"] = (
    df["home_true_strength"] -
    df["away_true_strength"]
)

## Dimensionality Reduction: Removing Redundant Features

Now that the critical information from the last 10 matches has been successfully extracted into dense, aggregated features (averages, differences, and ratios), the original match-by-match columns are completely redundant. 

Keeping them would unnecessarily increase the dimensionality of the dataset, slow down training times, and risk model overfitting. Therefore, we explicitly drop the raw historical columns for goals, ratings, locations, cup status, and rest days.

In [57]:
# Initialize a list to hold all the column names we want to drop
columns_to_drop = []

# Loop through both teams and all 10 historical match intervals
for team in ['home', 'away']:
    for i in range(1, 11):
        columns_to_drop.extend([
            f"{team}_club_recent_goals_scored_{i}",
            f"{team}_club_recent_goals_conceded_{i}",
            f"{team}_club_recent_team_rating_{i}",
            f"{team}_club_recent_played_at_home_{i}",
            f"{team}_club_recent_cup_game_{i}",
            f"{team}_days_since_match_{i}"
        ])

# Drop the columns from the dataframe
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

print(f"Removed {len(columns_to_drop)} redundant features.")
print(f"New dataset shape: {df.shape}")

Removed 120 redundant features.
New dataset shape: (110921, 94)


## Save Engineered Data
The engineered dataframe is saved and ready to be loaded into the modeling pipeline.

In [58]:
# df_selected.to_csv("../data/engineered_data.csv", index=False)
df.to_csv("../data/engineered_data.csv", index=False)
print(f"Feature engineering complete. Data matrix shape: {df.shape}")

Feature engineering complete. Data matrix shape: (110921, 94)
